In [ ]:
%pip install -U python-woc pandas matplotlib tqdm
# please clear the output of this cell in your notebook before checking it in

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ["apache/ctakes","Bioconductor/Biobase","open-sdr/openwifi","HARPgroup/r-dh-ecohydro","r-quantities/units","typelevel/frameless","arogozhnikov/einops","kelu124/echomods","stan-dev/loo","issp-center-dev/HPhi"]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,000271f3940e5c277416b7dd2ede108da8aac4a7,apache_ctakes


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  2%|▊                                       | 36/1766 [00:37<29:51,  1.04s/it]

Got Errors {'1256cce344f11ce68de8ec1cd57effe6fbb709df': 'Key 1256cce344f11ce68de8ec1cd57effe6fbb709df not found in /da5_fast/All.sha1c/commit_18.tch'}


  3%|█▎                                      | 57/1766 [00:58<29:23,  1.03s/it]

Got Errors {'1d48864ea30a469e44196d1df4d92b6e7af301d5': 'Key 1d48864ea30a469e44196d1df4d92b6e7af301d5 not found in /da5_fast/All.sha1c/commit_29.tch'}


  3%|█▎                                      | 59/1766 [01:00<29:22,  1.03s/it]

...

Got Errors {'82d58b07a4478cf71f8cd68d97e32e2aa97da1aa': 'Key 82d58b07a4478cf71f8cd68d97e32e2aa97da1aa not found in /da5_fast/All.sha1c/commit_2.tch'}


 91%|██████████████████████████████████▍   | 1601/1766 [27:41<02:50,  1.04s/it]

Got Errors {'53e9a5f2bbd700cea27e9d735e4ebd7d7f55ca2b': 'Key 53e9a5f2bbd700cea27e9d735e4ebd7d7f55ca2b not found in /da5_fast/All.sha1c/commit_83.tch'}


100%|██████████████████████████████████████| 1766/1766 [30:31<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,000271f3940e5c277416b7dd2ede108da8aac4a7,070248827887375dd013167482260bd68d487e5a,[1ade4dd32196aca1f542b9aa3599cff52f71c5d2],Sean Finan <seanfinan@apache.org>,1484780774,+0000,Sean Finan <seanfinan@apache.org>,1484780774,+0000,"Write XMIs in tree, write using addLast\n\ngit...",000271f3940e5c277416b7dd2ede108da8aac4a7,apache_ctakes
1,0016a61c5ab40ab4ba90b2e38be6ffab17d3bae9,a8522236c71bb35ed9f364807855a5354a34611d,[ecf32c5e3dcdfa619cf57e0f38e56add540aede0],Sean Patrick Murphy <spmurphy50@apache.org>,1318604266,+0000,Sean Patrick Murphy <spmurphy50@apache.org>,1318604266,+0000,Fixes for version 1.2.0.\n\ngit-svn-id: https:...,0016a61c5ab40ab4ba90b2e38be6ffab17d3bae9,apache_ctakes


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '000271f3940e5c277416b7dd2ede108da8aac4a7', 'tree': '070248827887375dd013167482260bd68d487e5a', 'parent': ['1ade4dd32196aca1f542b9aa3599cff52f71c5d2'], 'author': 'Sean Finan <seanfinan@apache.org>', 'author_time': 1484780774, 'author_tz': '+0000', 'committer': 'Sean Finan <seanfinan@apache.org>', 'committer_time': 1484780774, 'committer_tz': '+0000', 'message': 'Write XMIs in tree, write using addLast\n\ngit-svn-id: https://svn.apache.org/repos/asf/ctakes/trunk@1779398 13f79535-47bb-0310-9956-ffa450edef68\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])

commit_data = df_commit_data.to_dict("records")

for k in commit_data:
  row = pd.Series({'project': k['project'], 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,apache_ctakes,000271f3940e5c277416b7dd2ede108da8aac4a7,Sean Finan <seanfinan@apache.org>,1484780774,"Write XMIs in tree, write using addLast\n\ngit..."


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='dbritto3'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='w', header=['project_wocid', 'commit_sha1', 'author', 'time', 'commit message'])

In [9]:
project_stats = dfinf.groupby("project").agg(ncommits=("commit", "nunique"),nauthors=("author", "nunique"),earliest=("time", "min"),latest=("time", "max")).reset_index()

project_stats["earliest"] = pd.to_datetime(project_stats["earliest"], unit="s", utc=True)
project_stats["latest"] = pd.to_datetime(project_stats["latest"], unit="s", utc=True)

display(project_stats)

,project,ncommits,nauthors,earliest,latest
0,apache_ctakes,4914,61,2009-03-08 14:52:58+00:00,2025-09-26 02:51:31+00:00
1,arogozhnikov_einops,1011,64,2018-09-22 00:45:08+00:00,2025-08-17 03:11:57+00:00
2,bioconductor_biobase,1027,57,2001-10-04 15:55:43+00:00,2026-01-16 15:44:14+00:00
3,harpgroup_r-dh-ecohydro,475,16,2017-05-30 20:25:36+00:00,2025-05-02 16:50:20+00:00
4,issp-center-dev_hphi,2428,47,2015-09-11 02:40:52+00:00,2025-10-25 06:47:04+00:00
5,kelu124_echomods,1331,10,2016-04-12 18:57:33+00:00,2025-05-23 23:20:25+00:00
6,open-sdr_openwifi,1043,45,2019-11-08 11:07:21+00:00,2025-10-28 18:25:23+00:00
7,r-quantities_units,1130,50,2016-06-07 07:40:34+00:00,2025-10-10 08:14:18+00:00
8,stan-dev_loo,1607,45,2015-06-13 07:12:22+00:00,2025-11-04 06:55:26+00:00
9,typelevel_frameless,2630,109,2015-03-28 06:03:10+00:00,2025-11-03 20:31:19+00:00


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

*WoC returned 62 "Key not found" errors. The WoC statistics below show the commits that were successfully collected.

### apache_ctakes

GitHub:
- Stars: 138
- Forks: 27
- Last commit date: 2026-08-12

World of Code:
- Number of commits: 4914
- Number of authors: 61
- Earliest commit time: 2009-03-08 14:52:58
- Latest commit time: 2025-09-26 02:51:31

### arogozhnikov_einops

GitHub:
- Stars: 9605
- Forks: 402
- Last commit date: 2026-08-26

World of Code:
- Number of commits: 1011
- Number of authors: 64
- Earliest commit time: 2018-09-22 00:45:08
- Latest commit time: 2025-08-17 03:11:57

### bioconductor_biobase

GitHub:
- Stars: 9
- Forks: 11
- Last commit date: 2026-07-28

World of Code:
- Number of commits: 1027
- Number of authors: 57
- Earliest commit time: 2001-10-04 15:55:43
- Latest commit time: 2026-01-16 15:44:14

### harpgroup_r-dh-ecohydro

GitHub:
- Stars: 0
- Forks: 1
- Last commit date: 2025-05-02

World of Code:
- Number of commits: 475
- Number of authors: 16
- Earliest commit time: 2017-05-30 20:25:36
- Latest commit time: 2025-05-02 16:50:20

### issp-center-dev_hphi

GitHub:
- Stars: 104
- Forks: 29
- Last commit date: 2026-07-15

World of Code:
- Number of commits: 2428
- Number of authors: 47
- Earliest commit time: 2015-09-11 02:40:52
- Latest commit time: 2025-10-25 06:47:04

### kelu124_echomods

GitHub:
- Stars: 448
- Forks: 121
- Last commit date: 2026-04-08

World of Code:
- Number of commits: 1331
- Number of authors: 10
- Earliest commit time: 2016-04-12 18:57:33
- Latest commit time: 2025-05-23 23:20:25

### open-sdr_openwifi

GitHub:
- Stars: 4822
- Forks: 822
- Last commit date: 2026-09-21

World of Code:
- Number of commits: 1043
- Number of authors: 45
- Earliest commit time: 2019-11-08 11:07:21
- Latest commit time: 2025-10-28 18:25:23

### r-quantities_units

GitHub:
- Stars: 203
- Forks: 35
- Last commit date: 2026-09-22

World of Code:
- Number of commits: 1130
- Number of authors: 50
- Earliest commit time: 2016-06-07 07:40:34
- Latest commit time: 2025-10-10 08:14:18

### stan-dev_loo

GitHub:
- Stars: 157
- Forks: 39
- Last commit date: 2026-09-15

World of Code:
- Number of commits: 1607
- Number of authors: 45
- Earliest commit time: 2015-06-13 07:12:22
- Latest commit time: 2025-11-04 06:55:26

### typelevel_frameless

GitHub:
- Stars: 895
- Forks: 135
- Last commit date: 2026-08-28

World of Code:
- Number of commits: 2630
- Number of authors: 109
- Earliest commit time: 2015-03-28 06:03:10
- Latest commit time: 2025-11-03 20:31:19
